[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafamayo/Workshop-UPNA-2026/blob/main/labs/day1/lab1-basic-fhir-rest/lab1-starter-notebook.ipynb)

# Lab 1 Starter Notebook — Basic FHIR & REST
UPNA 2026 — Clinical Information Systems Workshop

This notebook contains starter code for interacting with a public FHIR server.

## Learning goals
- Send basic REST requests to a FHIR server
- Retrieve server metadata
- GET patients and observations
- Create a Patient resource (POST)
- Create an Observation linked to that Patient
- Retrieve resources using search parameters

⚠️ **Important:** Replace any placeholder values (e.g., `YOUR_PATIENT_ID`) with your own values returned by the server.

In [ ]:
!pip install requests

In [ ]:
import requests
import json

FHIR_SERVER = "https://hapi.fhir.org/baseR5/"  # replace if using another server
print("FHIR server set to:", FHIR_SERVER)

## 1. Check server metadata
This helps confirm that the FHIR server is reachable and gives an overview of supported features.

Run the cell below:

In [ ]:
metadata = requests.get(FHIR_SERVER + "metadata").json()
list(metadata.keys())[:10]

## 2. Get patients (GET)
Use GET to obtain a list of existing patients on the server

In [ ]:
requests.get(FHIR_SERVER + "Patient").json()

### 2.1 Search for patients by name (GET)
Use GET and the search syntax to obtain patients that match a specific query.

Search for patients with name "Smith"

In [ ]:
requests.get(FHIR_SERVER + "Patient/2202").json()

In [ ]:
requests.get(FHIR_SERVER + "Patient?name=smith").json()

In [ ]:
requests.get(FHIR_SERVER + "Observation?subject=Patient/2202").json()

## 2. Create a Patient (POST)
Modify fields as you like. Remember:
- `resourceType` is required
- Birthdates must follow the ISO format: `YYYY-MM-DD`

In [ ]:
patient = {
    "resourceType": "Patient",
    "name": [{"family": "Merino", "given": ["Maria"]}],
    "gender": "female",
    "birthDate": "1967-10-23"
}

response = requests.post(
    FHIR_SERVER + "Patient",
    headers={"Content-Type": "application/fhir+json"},
    data=json.dumps(patient)
)

print("Status:", response.status_code)
created_patient = response.json()
created_patient

### Extract the new Patient ID
Use it for later references.

In [ ]:
patient_id = created_patient.get("id")
patient_id

## 3. Create an Observation linked to the Patient
Example: Heart rate using LOINC code `8867-4`.

Replace `patient_id` with your extracted value if needed.

In [ ]:
observation = {
    "resourceType": "Observation",
    "status": "final",
    "code": {
        "coding": [{"system": "http://loinc.org", "code": "8867-4"}]
    },
    "subject": {"reference": f"Patient/{patient_id}"},
    "valueQuantity": {"value": 72, "unit": "bpm"}
}

response = requests.post(
    FHIR_SERVER + "Observation",
    headers={"Content-Type": "application/fhir+json"},
    data=json.dumps(observation)
)

print("Status:", response.status_code)
created_observation = response.json()
created_observation

Extract the Observation ID:

In [ ]:
obs_id = created_observation.get("id")
obs_id

## 4. Retrieve Observations for this Patient
Demonstrates a FHIR search query using REST.

In [ ]:
search_result = requests.get(
    FHIR_SERVER + f"Observation?subject=Patient/{patient_id}"
).json()

search_result

## 5. Explore: Try searching by code
Example: LOINC `8867-4` for heart rate.

In [ ]:
hr_result = requests.get(
    FHIR_SERVER + "Observation?code=8867-4"
).json()

hr_result

## 6. Exercise: What happens if you try to reference a nonexistent Patient?
- Try posting an Observation with `subject.reference = "Patient/XXXXX"` where `XXXXX` does not exist.
- Observe the error response.

Document your answer below:

In [ ]:
# Write notes or experiment here
response=requests.get(FHIR_SERVER + "Patient/999999")

print("Status:", response.status_code)
response.json()

### Add more Observations

In [ ]:
observation = {
    "resourceType": "Observation",
    "status": "final",
    "code": {
        "coding": [{"system": "http://loinc.org", "code": "8310-5"}]
    },
    "subject": {"reference": f"Patient/{patient_id}"},
    "valueQuantity": {"value": 37.5, "unit": "Cel"}
}

response = requests.post(
    FHIR_SERVER + "Observation",
    headers={"Content-Type": "application/fhir+json"},
    data=json.dumps(observation)
)

print("Status:", response.status_code)
created_observation = response.json()
created_observation

In [ ]:
search_result = requests.get(
    FHIR_SERVER + f"Observation?subject=Patient/{patient_id}"
).json()

search_result